In [ ]:
import os
import re
import json
import time
import requests
from datetime import datetime


def handle_rate_limit(response):
    if response.status_code == 403 and "X-RateLimit-Remaining" in response.headers:
        remaining = int(response.headers.get("X-RateLimit-Remaining", 0))
        reset_ts = int(response.headers.get("X-RateLimit-Reset", 0))
        now_ts = int(time.time())

        if remaining == 0:
            wait_time = reset_ts - now_ts + 5
            reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')
            print(f"API limit. Waiting {wait_time}s (reset on {reset_time} UTC)...")
            time.sleep(max(wait_time, 0))
            return True
    return False


def parse_pr_link(link):
    pattern = r"https://github\.com/([^/]+)/([^/]+)/pull/(\d+)"
    match = re.match(pattern, link.strip())
    if match:
        owner, repo, pr_number = match.groups()
        return owner, repo, int(pr_number)
    return None, None, None


TOKEN = None
with open(".env", "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("GITHUB_TOKEN="):
            TOKEN = line.strip().split("=", 1)[1]

if not TOKEN:
    raise ValueError("No GITHUB_TOKEN found in file .env")

TOKEN = TOKEN.strip().strip("'").strip('"')

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json"
}


input_path = "pr_links.txt"                   
output_path = "human_pr_inline_review_comments_raw.txt"  
batch_size = 5000


with open(input_path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

total_lines = len(lines)
print(f"Found {total_lines} link PRs. Batching {batch_size} per part...")



with open(output_path, "a", encoding="utf-8") as out:
    for i in range(0, total_lines, batch_size):
        batch = lines[i:i + batch_size]
        print(f"\Processing part {i // batch_size + 1} ({len(batch)} links)...")

        for line in batch:
            parts = line.split()
            if len(parts) != 2:
                print(f"Wrong format: {line}")
                continue

            pr_id, link = parts
            owner, repo, pr_number = parse_pr_link(link)
            if not owner:
                print(f"Wrong link: {link}")
                continue

            url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/comments"

            while True:
                response = requests.get(url, headers=headers)
                if handle_rate_limit(response):
                    continue

                if response.status_code == 401:
                    print(f"Error 401 - no authorization! Check token or privileges.")
                    break

                if response.status_code != 200:
                    print(f"Error {response.status_code} while fetching: {url}")
                    break

                comments = response.json()
                for comment in comments:
                    user_info = comment.get("user", {})
                    record = {
                        "id": comment.get("id"),
                        "pull_request_review_id": comment.get("pull_request_review_id"),
                        "user": user_info.get("login") if isinstance(user_info, dict) else None,
                        "user_type": user_info.get("type") if isinstance(user_info, dict) else None,
                        "diff_hunk": comment.get("diff_hunk"),
                        "path": comment.get("path"),
                        "position": comment.get("position"),
                        "original_position": comment.get("original_position"),
                        "commit_id": comment.get("commit_id"),
                        "original_commit_id": comment.get("original_commit_id"),
                        "body": comment.get("body"),
                        "pull_request_url": comment.get("pull_request_url"),
                        "created_at": comment.get("created_at"),
                        "updated_at": comment.get("updated_at"),
                        "in_reply_to_id": comment.get("in_reply_to_id"),
                    }
                    out.write(json.dumps(record, ensure_ascii=False) + "\n")
                break

        if i + batch_size < total_lines:
            print(f"Quick break 10s before part ({i + batch_size}/{total_lines})...")
            time.sleep(10)

print(f"\nCompleted. Data save to: {output_path}")


Znaleziono 6618 linków PR. Przetwarzanie po 5000 na partię...

Przetwarzanie partii 1 (5000 linków)...


/tmp/ipykernel_2032678/4174652754.py:17: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  reset_time = datetime.utcfromtimestamp(reset_ts).strftime('%Y-%m-%d %H:%M:%S')


⏳ Limit API osiągnięty. Czekam 1347 sek. (reset o 2025-10-29 09:15:58 UTC)...
Krótka przerwa 10 sekund przed kolejną partią (5000/6618)...

Przetwarzanie partii 2 (1618 linków)...
⏳ Limit API osiągnięty. Czekam 1574 sek. (reset o 2025-10-29 10:16:04 UTC)...

Zakończono. Dane zapisano do pliku: human_pr_inline_review_comments_raw.txt


In [ ]:
# Convert to parquet
import pandas as pd
import json

input_file = "human_pr_inline_review_comments_raw.txt"
output_file = "human_pr_review_comments.parquet"

records = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line.strip()))

df = pd.DataFrame(records)
df.to_parquet(output_file, index=False)

print(f"Writing data to parquet file: {output_file}")
print(f"Data count: {len(df)}")

Zapisano dane do pliku Parquet: human_pr_review_comments.parquet
Liczba rekordów: 9455
